In [2]:
import requests
import json
from datetime import datetime
import time
import random
import asyncio, aiohttp
import aiolimiter
from tqdm.auto import tqdm


/Users/jonasbjaerke/Projects/data_analysis_practice/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
_TOKEN_CACHE = None  # global cache

def get_session_token():
    global _TOKEN_CACHE
    if _TOKEN_CACHE:   # reuse valid token
        return _TOKEN_CACHE

    url = "https://bsky.social/xrpc/com.atproto.server.createSession"
    handle = "repostproj.bsky.social"
    app_password = "vyvc-xg5q-seda-utaz" 

    for _ in range(3):  # retry a few times on rate limit
        r = requests.post(url, json={"identifier": handle, "password": app_password})
        if r.status_code == 429:
            time.sleep(1)
            continue
        r.raise_for_status()
        _TOKEN_CACHE = r.json()["accessJwt"]
        return _TOKEN_CACHE

    raise RuntimeError("Failed to obtain token after retries")

In [4]:


def load_posts_dict(file_path):
    """
    Reads a .jsonl file where each line is a JSON object representing a post.
    Returns a dict mapping post_id (or URI) -> post data.

    Supports multiple Bluesky data formats:
      - post["uri"]
      - post["post"]["uri"]
      - post["id"]
    """
    posts_dict = {}
    bad_lines = 0

    with open(file_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                post = json.loads(line)
            except json.JSONDecodeError:
                bad_lines += 1
                continue

            # Try to find a reliable ID/URI key
            post_id = (
                post.get("uri")
                or post.get("post", {}).get("uri")
                or post.get("id")
            )

            posts_dict[post_id] = post

    print(f"Loaded {len(posts_dict)} posts from {file_path} ({bad_lines} bad lines skipped)")
    return posts_dict


In [5]:
posts_dict = load_posts_dict("tennis.jsonl")   

Loaded 7651 posts from tennis.jsonl (0 bad lines skipped)


In [6]:


API_URL = "https://public.api.bsky.app/xrpc/app.bsky.feed.getRepostedBy"


async def fetch_reposters(session, uri, limit=100):
    """Fetch reposters for one post URI safely."""
    params = {"uri": uri, "limit": limit}
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        async with session.get(API_URL, params=params, headers=headers, timeout=300) as r:
            if r.status != 200:
                print(f"HTTP {r.status} for {uri}")
                return []
            data = await r.json()
            return [u["did"] for u in data.get("repostedBy", [])]
    except Exception as e:
        print(f"Error fetching {uri}: {e}")
        return []


async def collect_reposter_dict(posts_dict, concurrency=25):
    """
    Takes a dict of posts:
        { post_uri: {...}, post_uri2: {...}, ... }

    Returns a dict mapping each reposter DID -> [list of post URIs they reposted].
    Only includes posts that have repostCount > 0.
    """
    posts_to_check, tasks = [], []

    connector = aiohttp.TCPConnector(limit_per_host=concurrency)
    async with aiohttp.ClientSession(connector=connector) as session:

        for post_uri, post in posts_dict.items():
            repost_count = post.get("repostCount")

            if repost_count > 0:
                posts_to_check.append(post_uri)
                tasks.append(fetch_reposters(session, post_uri))

        # Run all requests concurrently
        repost_lists = await asyncio.gather(*tasks, return_exceptions=True)

    # Merge results into reposter → [post_uris] mapping
    reposter_dict = {}
    for post_uri, reposters in zip(posts_to_check, repost_lists):
        if not reposters or isinstance(reposters, Exception):
            continue
        for reposter in reposters:
            reposter_dict.setdefault(reposter, []).append(post_uri)

    print(f"Processed {len(posts_to_check)} posts. Found {len(reposter_dict)} unique reposters.")
    return reposter_dict


In [7]:
repost_dict = await collect_reposter_dict(posts_dict)

Processed 336 posts. Found 536 unique reposters.


In [8]:
print(len(repost_dict))

536


In [23]:


FOLLOW_API = "https://public.api.bsky.app/xrpc/app.bsky.graph.getFollows"


def get_author_dids(posts_dict):
    """Extract all unique author DIDs from post data."""
    authors = set()
    for post in posts_dict.values():
        did = (
            post.get("author", {}).get("did")
            or post.get("post", {}).get("author", {}).get("did")
        )
        if did:
            authors.add(did)
    return authors


async def fetch_follows(session, reposter, author_set, retries=3, first_page_only=True):
    """Fetch follows for one reposter; retry on transient failures."""
    follows = set()
    cursor = None
    headers = {"User-Agent": "Mozilla/5.0"}

    delay = 1
    for attempt in range(retries):
        try:
            while True:
                params = {"actor": reposter, "limit": 100}
                if cursor:
                    params["cursor"] = cursor

                async with session.get(FOLLOW_API, params=params, headers=headers) as r:
                    if r.status != 200:
                        if 500 <= r.status < 600 and attempt < retries - 1:
                            await asyncio.sleep(delay)
                            delay *= 2
                            continue
                        print(f"⚠️ HTTP {r.status} for {reposter}")
                        return reposter, []

                    data = await r.json()
                    follows.update(
                        u["did"] for u in data.get("follows", [])
                        if u.get("did") in author_set
                    )

                    cursor = data.get("cursor")
                    if not cursor or first_page_only:
                        break
            return reposter, list(follows)

        except Exception as e:
            if attempt < retries - 1:
                await asyncio.sleep(delay)
                delay *= 2
                continue
            print(f" {reposter} failed after {retries} retries: {e}")
            return reposter, []


async def collect_reposter_followed_authors(reposter_dict, posts_dict,
                                            concurrency=100, reqs_per_sec=50):
    """
    Fetch who each reposter follows (intersected with author_set),
    concurrently and rate-limited for stability.
    """
    reposters = list(reposter_dict.keys())
    total = len(reposters)
    author_set = get_author_dids(posts_dict)
    counter = {"done": 0}

    # --- tuned network setup ---
    rate_limiter = aiolimiter.AsyncLimiter(reqs_per_sec, 1)  # 50 req/s max
    connector = aiohttp.TCPConnector(limit=300, limit_per_host=50, ttl_dns_cache=300)
    timeout = aiohttp.ClientTimeout(total=10, connect=5, sock_read=5)

    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        sem = asyncio.Semaphore(concurrency)

        async def limited_fetch(reposter):
            async with sem, rate_limiter:
                rep, authors = await fetch_follows(session, reposter, author_set)
                counter["done"] += 1
                if counter["done"] % 100 == 0 or counter["done"] == total:
                    print(f"\rProgress: {counter['done']}/{total} "
                          f"({counter['done']/total:.1%})", end="", flush=True)
                return rep, authors

        tasks = [limited_fetch(r) for r in reposters]
        responses = await asyncio.gather(*tasks)

    return {r: authors for r, authors in responses if authors}


In [24]:
followers = await collect_reposter_followed_authors(repost_dict,posts_dict)

Progress: 1189/1189 (100.0%)

In [34]:

def get_negative_instance(reposts_dict, posts_dict, user_did, qnt=1):
    """
    Returns the ID of a post that was NOT reposted by the given user.
    If the user reposted all posts or there are no posts, returns None.
    """
    all_posts = set(posts_dict.keys())
    user_reposts = set(reposts_dict.get(user_did, {}).keys() if isinstance(reposts_dict.get(user_did), dict)
                       else reposts_dict.get(user_did, []))
    
    not_reposted = list(all_posts - user_reposts)
    if not not_reposted:
        return None
    
    return random.sample(not_reposted,qnt)


In [33]:
print(get_negative_instance(repost_dict,posts_dict,next(iter(repost_dict), None)))

['at://did:plc:5o6lhctqmichdrr4cwk3iqg7/app.bsky.feed.post/3m3qjmqeolk2f', 'at://did:plc:zjy7i2sxdo3wdqq4rc3lebcj/app.bsky.feed.post/3m2wxo3wfas2d', 'at://did:plc:tqan6vniv6gk6h4iybjnhsup/app.bsky.feed.post/3m3qdhy5pgs2g', 'at://did:plc:wk45xh4fuqyuaqwsv3itmt7u/app.bsky.feed.post/3m3b33wbrr22t', 'at://did:plc:gu3bxkap2fifcbaencr32rll/app.bsky.feed.post/3m3qd2gmnws24']


In [136]:
import aiohttp, asyncio, time, random, aiolimiter, uvloop

asyncio.set_event_loop_policy(uvloop.EventLoopPolicy())

API_URL = "https://public.api.bsky.app/xrpc/app.bsky.feed.getAuthorFeed"


async def collect_reposter_repost_times(repost_dict,
                                        concurrency=300,
                                        reqs_per_sec=120,
                                        max_pages=15):
    """
    ⚡ Extreme-speed repost timestamp scraper.
    Tuned for 80–140 req/s stable throughput.
    """

    reposters = sorted(repost_dict.keys(),
                       key=lambda r: len(repost_dict[r]),
                       reverse=True)  # heavy-first = fastest

    n = len(reposters)
    start = time.time()
    counter = 0

    rate = aiolimiter.AsyncLimiter(reqs_per_sec, 1)

    # Faster connector
    connector = aiohttp.TCPConnector(
        limit=600,
        limit_per_host=300,
        ttl_dns_cache=1200,
        enable_cleanup_closed=True,
        ssl=False  # BIG SPEEDUP
    )

    timeout = aiohttp.ClientTimeout(total=6, sock_read=4)

    async def fetch_user(session, reposter, target_uris):
        nonlocal counter

        found = {}
        remaining = set(target_uris)
        cursor = None
        pages = 0
        backoff = 0.2  # MUCH faster

        while remaining and pages < max_pages:

            params = {
                "actor": reposter,
                "limit": 100,
                "filter": "posts_and_author_threads"
            }
            if cursor:
                params["cursor"] = cursor

            async with rate:
                try:
                    async with session.get(API_URL, params=params) as r:
                        if r.status == 429:
                            await asyncio.sleep(backoff)
                            backoff = min(backoff * 1.5, 2.5)
                            continue
                        if r.status != 200:
                            break
                        data = await r.json()
                except:
                    await asyncio.sleep(backoff)
                    continue

            pages += 1
            feed = data.get("feed", [])
            if not feed:
                break

            for item in feed:
                reason = item.get("reason")
                if not reason or reason.get("$type") != "app.bsky.feed.defs#reasonRepost":
                    continue
                uri = item["post"]["uri"]
                if uri in remaining:
                    found[uri] = reason.get("indexedAt")
                    remaining.remove(uri)

            cursor = data.get("cursor")
            if not cursor:
                break

        counter += 1
        if counter % 200 == 0 or counter == n:
            elapsed = time.time() - start
            speed = counter / elapsed
            eta = (n - counter) / speed if speed > 0 else 0
            print(f"\r[{counter}/{n}]  {speed:.1f} u/s   ETA {eta/60:.1f} min", end="", flush=True)

        return reposter, found

    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        sem = asyncio.Semaphore(concurrency)

        async def task(reposter):
            async with sem:
                return await fetch_user(session, reposter, repost_dict[reposter])

        results = await asyncio.gather(*(task(r) for r in reposters))

    print("\nDone.")
    elapsed = time.time() - start
    print(f"⏱ {n} users in {elapsed:.1f}s  ({n/elapsed:.1f} u/s avg)")

    return {r: posts for r, posts in results if posts}


In [137]:
results = await collect_reposter_repost_times(repost_dict)


[8574/8574]  31.8 u/s   ETA 0.0 min
Done.
⏱ 8574 users in 269.4s  (31.8 u/s avg)


In [140]:
print(len(results))

8497


In [ ]:
with open("posts_dict.json", "w", encoding="utf-8") as f:
    json.dump(posts_dict, f, ensure_ascii=False, indent=2)
    
with open("repost_with_time.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

with open("reposts_dict.json", "w", encoding="utf-8") as f:
    json.dump(repost_dict, f, ensure_ascii=False, indent=2)

with open("followers_dict.json", "w", encoding="utf-8") as f:
    json.dump(followers, f, ensure_ascii=False, indent=2)

    

In [144]:
with open("repost_with_time.json", "r", encoding="utf-8") as f:
    repost_dict_times = json.load(f)

# Load repost_dict
with open("reposts_dict.json", "r", encoding="utf-8") as f:
    repost_dict = json.load(f)

# Load followers_dict
with open("followers_dict.json", "r", encoding="utf-8") as f:
    followers = json.load(f)

In [145]:


print(len(repost_dict["did:plc:q5ja22cc72afmkyq5t4xrhzt"]))

print(len(repost_dict_times["did:plc:q5ja22cc72afmkyq5t4xrhzt"]))

295
271


In [146]:
print(len(repost_dict), len(repost_dict_times), len(followers))


8574 8497 6203
